# Gradient Descent and Polynomial Regression

## Overview
In this lab, you will learn how to implement and compare three core optimization strategies used in machine learning:

- **Batch Gradient Descent (BGD)**
- **Stochastic Gradient Descent (SGD)**
- **Mini-Batch Gradient Descent (MBGD)**

You will first study the underlying mathematics and implement these methods **from scratch with NumPy**.
Then you will apply them to **polynomial regression**, analyze their behavior, and complete hands-on exercises.

## Learning Objectives
By the end of this lab, you should be able to:

1. Explain how gradient descent minimizes a cost function.
2. Derive and implement update rules for BGD, SGD, and MBGD.
3. Build polynomial features for nonlinear curve fitting.
4. Compare optimization methods in terms of speed, stability, and final performance.
5. Critically evaluate hyperparameters such as learning rate and batch size.

In [ ]:
# =============================
# Imports and visual settings
# =============================
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(42)

print("Libraries imported successfully.")

## 1) Gradient Descent: Concept and Mathematics

### Why Gradient Descent?
For many machine learning models, we define a loss (or cost) function and want to find model parameters that minimize it.
Gradient descent is an iterative optimization algorithm that updates parameters in the direction of the **negative gradient** of the loss.

### Linear Model Form
We use a hypothesis:

$$
\hat{y} = X\theta
$$

where:
- $X \in \mathbb{R}^{m \times n}$ is the design matrix,
- $\theta \in \mathbb{R}^{n \times 1}$ are model parameters,
- $\hat{y} \in \mathbb{R}^{m \times 1}$ are predictions.

### Cost Function (Mean Squared Error)

$$
J(\theta) = \frac{1}{2m} \sum_{i=1}^{m}(\hat{y}^{(i)} - y^{(i)})^2
$$

In matrix form:

$$
J(\theta) = \frac{1}{2m}(X\theta - y)^T(X\theta - y)
$$

### Gradient of MSE

$$
\nabla_\theta J(\theta) = \frac{1}{m}X^T(X\theta - y)
$$

### Update Rule

$$
\theta := \theta - \eta \nabla_\theta J(\theta)
$$

where $\eta$ is the learning rate.

---

## Three Variants

1. **Batch GD**: Uses all training examples each update.
   $$
   \theta := \theta - \eta \cdot \frac{1}{m}X^T(X\theta - y)
   $$

2. **Stochastic GD (SGD)**: Uses one sample at a time.
   $$
   \theta := \theta - \eta \cdot x_i^T(x_i\theta - y_i)
   $$

3. **Mini-Batch GD**: Uses a small batch $B$ each update.
   $$
   \theta := \theta - \eta \cdot \frac{1}{|B|}X_B^T(X_B\theta - y_B)
   $$

Trade-off intuition:
- Batch: stable but can be slow per update.
- SGD: fast, noisy updates.
- Mini-Batch: balanced and widely used in practice.

In [ ]:
class GradientDescentRegressor:
    """
    Linear regressor optimized using batch, stochastic, or mini-batch GD.
    """
    def __init__(self, method="batch", lr=0.01, epochs=200, batch_size=32, random_state=42):
        self.method = method
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.theta = None
        self.loss_history = []
        self.training_time = None

    def _compute_gradient(self, X, y):
        """Gradient of MSE cost with respect to theta."""
        m = X.shape[0]
        errors = X @ self.theta - y
        grad = (X.T @ errors) / m
        return grad

    def fit(self, X, y):
        """Train model parameters using selected GD method."""
        rng = np.random.default_rng(self.random_state)
        m, n = X.shape
        self.theta = np.zeros((n, 1))
        self.loss_history = []

        start = time.perf_counter()

        for epoch in range(self.epochs):
            if self.method == "batch":
                grad = self._compute_gradient(X, y)
                self.theta -= self.lr * grad

            elif self.method == "sgd":
                # Shuffle indices each epoch for better stochastic behavior.
                indices = rng.permutation(m)
                for i in indices:
                    Xi = X[i:i+1]
                    yi = y[i:i+1]
                    grad = Xi.T @ (Xi @ self.theta - yi)  # batch size is 1
                    self.theta -= self.lr * grad

            elif self.method == "mini-batch":
                indices = rng.permutation(m)
                for start_idx in range(0, m, self.batch_size):
                    batch_idx = indices[start_idx:start_idx + self.batch_size]
                    Xb = X[batch_idx]
                    yb = y[batch_idx]
                    grad = (Xb.T @ (Xb @ self.theta - yb)) / Xb.shape[0]
                    self.theta -= self.lr * grad

            else:
                raise ValueError("method must be 'batch', 'sgd', or 'mini-batch'")

            # Track full-dataset loss after each epoch.
            y_pred_epoch = X @ self.theta
            epoch_loss = mean_squared_error(y, y_pred_epoch)
            self.loss_history.append(epoch_loss)

        end = time.perf_counter()
        self.training_time = end - start
        return self

    def predict(self, X):
        """Predict targets for input matrix X."""
        return X @ self.theta

## 2) Simple Example: Compare BGD, SGD, and MBGD

We start with a simple linear dataset to compare convergence behavior of the three methods.

Model: $y = 4 + 3x + \epsilon$

In [ ]:
# Generate synthetic linear data
m = 250
X_raw = 2 * np.random.rand(m, 1)
noise = np.random.randn(m, 1) * 0.4
y = 4 + 3 * X_raw + noise

# Visualize dataset immediately after generation
plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y, alpha=0.6, s=25, color="teal")
plt.title("Generated Linear Dataset")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

# Add bias term for linear model
X = np.c_[np.ones((X_raw.shape[0], 1)), X_raw]

# Train all three variants
models = {
    "Batch GD": GradientDescentRegressor(method="batch", lr=0.08, epochs=50),
    "SGD": GradientDescentRegressor(method="sgd", lr=0.01, epochs=50),
    "Mini-Batch GD": GradientDescentRegressor(method="mini-batch", lr=0.03, epochs=50, batch_size=16)
}

results_simple = []

for name, model in models.items():
    model.fit(X, y)
    y_pred = model.predict(X)
    results_simple.append({
        "Method": name,
        "Final MSE": mean_squared_error(y, y_pred),
        "Training Time (s)": model.training_time,
        "Final Theta": model.theta.ravel()
    })

simple_results_df = pd.DataFrame(results_simple)
simple_results_df

In [ ]:
# Plot loss curves to compare convergence
plt.figure(figsize=(10, 5))
for name, model in models.items():
    plt.plot(model.loss_history, label=name, linewidth=2)

plt.title("Loss Curves on Simple Linear Data", fontsize=14)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Visualize final trained regression lines on top of the dataset
x_line = np.linspace(X_raw.min(), X_raw.max(), 300).reshape(-1, 1)
X_line = np.c_[np.ones((x_line.shape[0], 1)), x_line]

plt.figure(figsize=(9, 5))
plt.scatter(X_raw, y, alpha=0.5, s=22, color="gray", label="Dataset")
for name, model in models.items():
    y_line_pred = model.predict(X_line)
    plt.plot(x_line, y_line_pred, linewidth=2.4, label=f"{name} fit")

plt.title("Final Trained Regression Lines on Dataset")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.tight_layout()
plt.show()

display(simple_results_df.sort_values("Final MSE"))

### Interpretation
- **Batch GD** usually gives smoother convergence because each update uses the full dataset.
- **SGD** often converges quickly at first but exhibits noisier loss curves.
- **Mini-Batch GD** balances stability and computational efficiency.

In practice, mini-batch methods are widely used because they are GPU-friendly and often converge well.

## 3) Polynomial Regression from Scratch

### Why Polynomial Features?
A linear model in the original feature space can only fit straight lines/planes.
If the relationship between $x$ and $y$ is nonlinear, we can map input $x$ to polynomial features:

$$
\phi(x) = [x, x^2, x^3, \dots, x^d]
$$

Then fit a linear model in transformed space:

$$
\hat{y} = \theta_0 + \theta_1 x + \theta_2 x^2 + \cdots + \theta_d x^d
$$

This is still linear in parameters $\theta$, so gradient descent applies directly.

In [ ]:
# Generate synthetic polynomial data
m_poly = 300
x_poly = np.random.uniform(-3, 3, size=(m_poly, 1))
y_poly = 1.5 - 2.0 * x_poly + 0.9 * (x_poly ** 2) + 0.2 * (x_poly ** 3) + np.random.randn(m_poly, 1) * 1.2

# Visualize dataset immediately after generation
plt.figure(figsize=(8, 5))
plt.scatter(x_poly, y_poly, alpha=0.55, s=22, color="slateblue")
plt.title("Generated Polynomial Dataset")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

# Create polynomial features up to degree 3 using sklearn
degree = 3
poly = PolynomialFeatures(degree=degree, include_bias=False)
X_poly_raw = poly.fit_transform(x_poly)

# Standardize polynomial features for stable GD updates using sklearn
scaler = StandardScaler()
X_poly_scaled = scaler.fit_transform(X_poly_raw)

# Add explicit bias for our from-scratch GD implementation
X_poly_design = np.c_[np.ones((X_poly_scaled.shape[0], 1)), X_poly_scaled]

print("Polynomial data prepared.")
print("Design matrix shape:", X_poly_design.shape)

In [ ]:
# Train polynomial regression with all three GD variants
poly_models = {
    "Batch GD": GradientDescentRegressor(method="batch", lr=0.05, epochs=250),
    "SGD": GradientDescentRegressor(method="sgd", lr=0.005, epochs=250),
    "Mini-Batch GD": GradientDescentRegressor(method="mini-batch", lr=0.02, epochs=250, batch_size=32)
}

poly_results = []
for name, model in poly_models.items():
    model.fit(X_poly_design, y_poly)
    y_pred = model.predict(X_poly_design)

    # Convergence criterion example: first epoch where loss <= 1.05 * final loss
    final_loss = model.loss_history[-1]
    threshold = 1.05 * final_loss
    epochs_to_converge = next((i + 1 for i, l in enumerate(model.loss_history) if l <= threshold), model.epochs)

    poly_results.append({
        "Method": name,
        "Final MSE": mean_squared_error(y_poly, y_pred),
        "Epochs to Converge": epochs_to_converge,
        "Training Time (s)": model.training_time
    })

poly_results_df = pd.DataFrame(poly_results)
poly_results_df

In [ ]:
# Visualize fitted curves and learning curves
x_grid = np.linspace(x_poly.min(), x_poly.max(), 400).reshape(-1, 1)
X_grid_raw = poly.transform(x_grid)
X_grid_scaled = scaler.transform(X_grid_raw)
X_grid_design = np.c_[np.ones((X_grid_scaled.shape[0], 1)), X_grid_scaled]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: data + fitted curves
axes[0].scatter(x_poly, y_poly, alpha=0.35, s=18, label="Data")
for name, model in poly_models.items():
    y_grid_pred = model.predict(X_grid_design)
    axes[0].plot(x_grid, y_grid_pred, linewidth=2.5, label=name)

axes[0].set_title("Polynomial Regression Fits (Degree = 3)")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

# Right: learning curves
for name, model in poly_models.items():
    axes[1].plot(model.loss_history, linewidth=2, label=name)

axes[1].set_title("Learning Curves (Loss vs Epochs)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

display(poly_results_df.sort_values("Final MSE"))

### Observations
- All three methods can fit polynomial regression when features are engineered properly.
- Feature scaling is critical for numerical stability, especially for high-degree polynomial terms.
- SGD may oscillate more due to noisy updates, while Batch GD is smoother.
- Mini-Batch GD often gives a strong practical compromise.

# 4) Lab Exercises (Total: 100 points)

Complete the following tasks. Write clean code, include plots, and explain your findings in markdown.

## Task 1 (25 points)
**Implement Batch Gradient Descent from scratch** for a 4nd-degree polynomial regression on a new synthetic dataset (use different coefficients and noise from tutorial).

### Requirements
- Generate your own dataset.
- Build polynomial features (degree 4).
- Train using Batch GD.
- Plot fitted curve and learning curve.

In [ ]:

# ============================================================
# Task 1 – Batch Gradient Descent on Degree-4 Polynomial Data
# ============================================================

# --------------------------------------------------
# 1. Generate a NEW synthetic dataset
#    True function: y = 2 - 1.5x + 0.7x² - 0.4x³ + 0.15x⁴ + noise
#    Different coefficients AND noise level from the tutorial.
# --------------------------------------------------
np.random.seed(7)
m_t1 = 350                                      # more samples than tutorial (300)
x_t1 = np.random.uniform(-4, 4, size=(m_t1, 1)) # wider range than tutorial (−3,3)
noise_t1 = np.random.randn(m_t1, 1) * 2.0       # noise std = 2.0 (tutorial used 1.2)

y_t1 = (2.0
        - 1.5  * x_t1
        + 0.7  * x_t1**2
        - 0.4  * x_t1**3
        + 0.15 * x_t1**4
        + noise_t1)

# Quick visual of the raw dataset
plt.figure(figsize=(8, 5))
plt.scatter(x_t1, y_t1, alpha=0.55, s=22, color="darkorange")
plt.title("Task 1 – Generated Degree-4 Dataset")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 2. Build degree-4 polynomial features and standardise
# --------------------------------------------------
degree_t1 = 4
poly_t1  = PolynomialFeatures(degree=degree_t1, include_bias=False)
X_t1_raw = poly_t1.fit_transform(x_t1)           # shape (m, 4)

scaler_t1     = StandardScaler()
X_t1_scaled   = scaler_t1.fit_transform(X_t1_raw) # zero mean, unit variance

# Prepend explicit bias column for our GradientDescentRegressor
X_t1_design   = np.c_[np.ones((X_t1_scaled.shape[0], 1)), X_t1_scaled]

print("Design matrix shape:", X_t1_design.shape)   # (350, 5)

# --------------------------------------------------
# 3. Train Batch Gradient Descent
# --------------------------------------------------
bgd_t1 = GradientDescentRegressor(
    method="batch",
    lr=0.05,
    epochs=300,
    random_state=7
)
bgd_t1.fit(X_t1_design, y_t1)

y_pred_t1 = bgd_t1.predict(X_t1_design)
print(f"Task 1 – Batch GD Final MSE : {mean_squared_error(y_t1, y_pred_t1):.4f}")
print(f"Task 1 – Batch GD θ        : {bgd_t1.theta.ravel()}")

# --------------------------------------------------
# 4. Plots: (a) Fitted curve  (b) Learning curve
# --------------------------------------------------
# Smooth curve for plotting
x_grid_t1     = np.linspace(x_t1.min(), x_t1.max(), 500).reshape(-1, 1)
X_grid_t1_raw = poly_t1.transform(x_grid_t1)
X_grid_t1_sc  = scaler_t1.transform(X_grid_t1_raw)
X_grid_t1_des = np.c_[np.ones((X_grid_t1_sc.shape[0], 1)), X_grid_t1_sc]
y_grid_t1     = bgd_t1.predict(X_grid_t1_des)

# True underlying function (noise-free)
y_true_t1 = (2.0 - 1.5*x_grid_t1 + 0.7*x_grid_t1**2
             - 0.4*x_grid_t1**3 + 0.15*x_grid_t1**4)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# — Left: fitted curve vs data
axes[0].scatter(x_t1, y_t1, alpha=0.40, s=18, color="darkorange", label="Data")
axes[0].plot(x_grid_t1, y_true_t1, "k--", linewidth=1.8, label="True function")
axes[0].plot(x_grid_t1, y_grid_t1, "b-", linewidth=2.5, label="Batch GD fit (deg-4)")
axes[0].set_title("Task 1 – Fitted Polynomial Curve (Degree 4)")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

# — Right: learning curve
axes[1].plot(bgd_t1.loss_history, color="royalblue", linewidth=2)
axes[1].set_title("Task 1 – Batch GD Learning Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Loss")

plt.tight_layout()
plt.show()


## Task 2 (25 points)
**Implement Stochastic Gradient Descent (SGD)** and tune the learning rate.

### Requirements
- Use one polynomial dataset.
- Try multiple learning rates (e.g., 0.0005, 0.001, 0.005, 0.01).
- Plot loss curves for each learning rate.
- Explain underfitting/divergence/instability if observed.

In [ ]:

# ============================================================
# Task 2 – SGD: Effect of Learning Rate
# ============================================================
# Re-use the degree-4 dataset and feature pipeline from Task 1
# (x_t1, y_t1, X_t1_design, poly_t1, scaler_t1)

learning_rates = [0.0005, 0.001, 0.005, 0.01]
sgd_histories  = {}
sgd_metrics    = []

for lr in learning_rates:
    model = GradientDescentRegressor(
        method="sgd",
        lr=lr,
        epochs=250,
        random_state=7
    )
    model.fit(X_t1_design, y_t1)
    y_pred_lr = model.predict(X_t1_design)

    sgd_histories[lr] = model.loss_history
    sgd_metrics.append({
        "Learning Rate": lr,
        "Final MSE"    : round(mean_squared_error(y_t1, y_pred_lr), 4),
        "Min MSE"      : round(min(model.loss_history), 4),
        "Training Time (s)": round(model.training_time, 4)
    })

# — Plot all loss curves on one figure
plt.figure(figsize=(11, 6))
for lr, history in sgd_histories.items():
    plt.plot(history, linewidth=2, label=f"lr = {lr}")

plt.title("Task 2 – SGD Loss Curves for Different Learning Rates")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()

# — Metric table
sgd_metrics_df = pd.DataFrame(sgd_metrics)
display(sgd_metrics_df)



### Task 2 Analysis – Learning Rate Effects on SGD

| Learning rate | Observed behaviour |
|---|---|
| **0.0005** | Very slow learning. The loss decreases steadily but makes minimal progress in 250 epochs — the model is effectively **underfitting** because the step size is too small to traverse the loss surface in time. |
| **0.001** | Slower but stable convergence. Still approaches a good solution, just more slowly than necessary. |
| **0.005** | Balanced: the loss drops quickly and settles near the optimal value. This is close to the sweet spot for this dataset / feature scale. |
| **0.01** | Fast initial drop, but the curve shows clear **oscillation / instability** because each noisy sample pushes the parameters too far. The final MSE may still be acceptable, but the trajectory is jagged and the model risks overshooting the minimum. |

**Key takeaways**
- Too small a learning rate → slow convergence / practical underfitting within a fixed epoch budget.
- Too large a learning rate → noisy, oscillating updates that can prevent convergence or even diverge.
- Feature standardisation (applied in pre-processing) greatly narrows the safe range of learning rates, making tuning easier.


## Task 3 (25 points)
**Implement Mini-Batch Gradient Descent** and study different batch sizes.

### Requirements
- Try batch sizes: 8, 16, 32, 64.
- Keep other settings fixed (same learning rate, epochs, dataset).
- Plot loss curves for each batch size.
- Analyze trade-off: convergence speed vs stability.

In [ ]:

# ============================================================
# Task 3 – Mini-Batch GD: Effect of Batch Size
# ============================================================
# Reuse the degree-4 dataset and design matrix from Task 1
# (X_t1_design, y_t1)  ← same data, same pre-processing

batch_sizes   = [8, 16, 32, 64]
mbgd_histories = {}
mbgd_metrics   = []

FIXED_LR     = 0.02    # same for all batch sizes
FIXED_EPOCHS = 250     # same for all batch sizes

for bs in batch_sizes:
    model = GradientDescentRegressor(
        method="mini-batch",
        lr=FIXED_LR,
        epochs=FIXED_EPOCHS,
        batch_size=bs,
        random_state=7
    )
    model.fit(X_t1_design, y_t1)
    y_pred_bs = model.predict(X_t1_design)

    mbgd_histories[bs] = model.loss_history
    mbgd_metrics.append({
        "Batch Size"        : bs,
        "Final MSE"         : round(mean_squared_error(y_t1, y_pred_bs), 4),
        "Min MSE"           : round(min(model.loss_history), 4),
        "Training Time (s)" : round(model.training_time, 4)
    })

# — Loss curves
plt.figure(figsize=(11, 6))
for bs, history in mbgd_histories.items():
    plt.plot(history, linewidth=2, label=f"Batch size = {bs}")

plt.title("Task 3 – Mini-Batch GD Loss Curves for Different Batch Sizes")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()

# — Metric table
mbgd_metrics_df = pd.DataFrame(mbgd_metrics)
display(mbgd_metrics_df)



### Task 3 Analysis – Batch Size Trade-off

| Batch size | Convergence speed | Stability | Notes |
|---|---|---|---|
| **8** | Fastest drop in early epochs | Noisy / jagged loss curve | Small batches → high gradient variance, similar to SGD |
| **16** | Fast | Moderate noise | Good balance for small datasets |
| **32** | Moderate | Smoother | Industry-standard default; GPU-friendly |
| **64** | Slowest (fewer updates per epoch) | Smoothest | Approaches Batch GD behaviour; may need more epochs |

**Trade-off summary**

- **Smaller batch sizes** perform more gradient updates per epoch (one update every `bs` samples), so each epoch covers the loss surface more thoroughly — at the cost of high variance in each gradient estimate.
- **Larger batch sizes** produce lower-variance (but fewer) updates per epoch, resulting in smoother but potentially slower convergence.
- For this dataset (350 samples, degree-4 features), `batch_size=16` or `32` offers the best balance: fast convergence without too much oscillation.
- Wall-clock training time grows slightly with smaller batches due to the overhead of more Python loop iterations, even though each iteration is cheaper mathematically.


## Task 4 (25 points)
**Compare Batch, SGD, and Mini-Batch GD** on the same polynomial regression dataset.

### Required comparison outputs
- Final MSE
- Number of epochs to converge
- Training time
- Final fitted curve quality (visual inspection)

Present a summary table and at least one comparison plot.

In [ ]:

# ============================================================
# Task 4 – Full Comparison: Batch GD vs SGD vs Mini-Batch GD
# ============================================================
# Shared dataset: Task 1 data (x_t1, y_t1, X_t1_design)
# All methods use the same degree-4 scaled feature matrix and
# the same number of epochs so comparison is fair.

SHARED_EPOCHS = 250

task4_models = {
    "Batch GD"    : GradientDescentRegressor(method="batch",      lr=0.05,  epochs=SHARED_EPOCHS,                   random_state=7),
    "SGD"         : GradientDescentRegressor(method="sgd",        lr=0.005, epochs=SHARED_EPOCHS,                   random_state=7),
    "Mini-Batch GD": GradientDescentRegressor(method="mini-batch", lr=0.02,  epochs=SHARED_EPOCHS, batch_size=32,    random_state=7),
}

task4_rows = []

for name, model in task4_models.items():
    model.fit(X_t1_design, y_t1)
    y_pred_t4 = model.predict(X_t1_design)
    final_mse = mean_squared_error(y_t1, y_pred_t4)

    # Convergence epoch: first epoch where loss ≤ 1.05 × final loss
    threshold_t4 = 1.05 * model.loss_history[-1]
    epochs_to_conv = next(
        (i + 1 for i, l in enumerate(model.loss_history) if l <= threshold_t4),
        SHARED_EPOCHS
    )

    task4_rows.append({
        "Method"              : name,
        "Final MSE"           : round(final_mse, 4),
        "Min Loss (any epoch)": round(min(model.loss_history), 4),
        "Epochs to Converge"  : epochs_to_conv,
        "Training Time (s)"   : round(model.training_time, 4),
    })

summary_df = pd.DataFrame(task4_rows)

# ---- Plot 1: Loss curves comparison ----
plt.figure(figsize=(11, 5))
for name, model in task4_models.items():
    plt.plot(model.loss_history, linewidth=2, label=name)
plt.title("Task 4 – Learning Curves: Batch GD vs SGD vs Mini-Batch GD")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 2: Fitted curves on the same graph ----
x_grid_t4     = np.linspace(x_t1.min(), x_t1.max(), 500).reshape(-1, 1)
X_grid_t4_raw = poly_t1.transform(x_grid_t4)
X_grid_t4_sc  = scaler_t1.transform(X_grid_t4_raw)
X_grid_t4_des = np.c_[np.ones((X_grid_t4_sc.shape[0], 1)), X_grid_t4_sc]

y_true_t4 = (2.0 - 1.5*x_grid_t4 + 0.7*x_grid_t4**2
             - 0.4*x_grid_t4**3 + 0.15*x_grid_t4**4)

plt.figure(figsize=(11, 6))
plt.scatter(x_t1, y_t1, alpha=0.35, s=16, color="gray", label="Data")
plt.plot(x_grid_t4, y_true_t4, "k--", linewidth=1.8, label="True function")

colors = ["royalblue", "tomato", "seagreen"]
for (name, model), color in zip(task4_models.items(), colors):
    y_grid_t4 = model.predict(X_grid_t4_des)
    plt.plot(x_grid_t4, y_grid_t4, linewidth=2.5, color=color, label=f"{name} fit")

plt.title("Task 4 – Fitted Polynomial Curves (Degree 4) — All Methods")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.tight_layout()
plt.show()

# ---- Summary table ----
display(summary_df.sort_values("Final MSE").reset_index(drop=True))


# 5) Bonus Questions (Theoretical - 10 points)

Answer the following conceptual questions in your own words.


### 1) (4 points)
**Explain the bias-variance tradeoff in the context of choosing the degree of the polynomial. What happens if the degree is too low or too high?**

**Your answer:**

The **bias-variance tradeoff** describes a fundamental tension between two sources of error in a learned model:

- **Bias** is the systematic error caused by overly simplistic assumptions. A model with high bias consistently misses the true pattern.
- **Variance** is the sensitivity of the model to small changes in the training data. A model with high variance fits the training data too closely and performs poorly on unseen data.

**Polynomial degree and the tradeoff:**

| Degree | Bias | Variance | Behaviour |
|---|---|---|---|
| Too low (e.g., degree 1 for a cubic relationship) | High | Low | **Underfitting** – the model cannot express the true curvature; training and test errors are both large. |
| Just right (matches the true complexity) | Low | Low | Best generalisation – low training error and low test error. |
| Too high (e.g., degree 10 for a cubic relationship) | Low | High | **Overfitting** – the model memorises noise; training error is tiny, but test error is large; the curve oscillates wildly, especially near the data boundaries (Runge's phenomenon). |

In practice we select the polynomial degree via cross-validation, choosing the value that minimises validation error — the sweet spot between underfitting and overfitting.



### 2) (3 points)
**Why does Stochastic Gradient Descent have higher variance in parameter updates compared to Batch Gradient Descent? How does Mini-Batch GD try to balance this?**

**Your answer:**

**Why SGD has higher variance:**

In Batch GD the gradient is computed over **all m training examples**:

$$\nabla_\theta J = \frac{1}{m} X^T(X\theta - y)$$

This is the exact gradient of the cost function. With many samples the individual errors average out, so the direction and magnitude of each update are highly stable.

In SGD the gradient uses only **a single randomly chosen sample**:

$$\nabla_\theta J \approx x_i^T(x_i\theta - y_i)$$

One sample is an extremely noisy estimate of the true gradient. Different samples point in different (often contradictory) directions, so successive parameter updates vary wildly. This produces the characteristic jagged loss curves seen in SGD.

**How Mini-Batch GD balances the two extremes:**

Mini-Batch GD averages the gradient over a small subset (batch) of $B$ samples:

$$\nabla_\theta J \approx \frac{1}{B} X_B^T(X_B\theta - y_B)$$

By the law of large numbers, averaging over $B$ examples reduces the variance of the gradient estimate by a factor of roughly $B$ compared to SGD ($B=1$), while still being far cheaper per update than full-batch GD ($B=m$).

The result is:
- Smoother updates than SGD (lower variance), so the loss curve is less noisy.
- More updates per epoch than Batch GD (more exploration of the loss surface per unit time).
- GPU-parallelisable: batch matrix operations are highly optimised on modern hardware.

This is why mini-batch GD (typically $B=32$–$256$) is the default optimisation strategy in deep learning.



### 3) (3 points)
**Discuss the importance of feature scaling when using Gradient Descent for polynomial regression. What problems can arise without scaling?**

**Your answer:**

**Why scaling matters — the loss surface perspective:**

Gradient descent moves the parameters in proportion to the gradient of the loss. When features have very different magnitudes, the loss surface becomes an elongated, ill-conditioned "valley" rather than a round bowl. Updates oscillate across the narrow axis (large-magnitude feature) while making tiny progress along the wide axis (small-magnitude feature), forcing us to use a tiny learning rate to avoid divergence — and slowing convergence dramatically.

**The problem is magnified for polynomial features:**

For a single raw feature $x \in [-4, 4]$, the polynomial features are:

| Feature | Typical magnitude |
|---|---|
| $x$ | $\sim 4$ |
| $x^2$ | $\sim 16$ |
| $x^3$ | $\sim 64$ |
| $x^4$ | $\sim 256$ |

Without scaling, $x^4$ is **64× larger** than $x$. This creates a severely skewed loss landscape. Gradient descent will:
1. **Diverge or oscillate** for any learning rate large enough to move $\theta$ for the small-magnitude features.
2. **Require an extremely small learning rate** to handle the large features, causing the small-magnitude parameters to converge unacceptably slowly (practical underfitting).
3. **Produce numerically unstable gradients** because intermediate products like $x^4 \cdot \theta_4$ can overflow or dominate the gradient computation.

**How standardisation (zero mean, unit variance) fixes this:**

After `StandardScaler`, all features have mean ≈ 0 and variance ≈ 1. The loss surface becomes much more isotropic, a single learning rate works well across all parameter dimensions, and convergence is faster and more stable. This is why we always call `StandardScaler` before fitting gradient-descent-based models on polynomial features.


# 6) Conclusion

In this lab, you:
- Implemented **Batch GD**, **SGD**, and **Mini-Batch GD** from scratch using NumPy.
- Compared optimization behavior through loss curves and empirical metrics.
- Applied gradient descent methods to **polynomial regression** with feature engineering.
- Explored practical tuning effects of learning rate and batch size.

Key takeaway: there is no single universally best optimizer setup. Effective training depends on balancing learning rate, batch size, feature scaling, and computational constraints.